In [1]:
# genetic algorithm search of the one max optimization problem
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import randint
from numpy.random import rand
import scipy.special, numpy


Genetic Algorithm Functions

In [2]:

# tournament selection
def selection(pop, scores, k=3):
	# first random selection
	selection_ix = randint(len(pop))
	for ix in randint(0, len(pop), k-1):
		# check if better (e.g. perform a tournament)
		if scores[ix] < scores[selection_ix]:
			selection_ix = ix
	return pop[selection_ix]

In [3]:

# crossover two parents to create two children
def crossover(p1, p2, r_cross):
	# children are copies of parents by default
	c1, c2 = p1.copy(), p2.copy()
	# check for recombination
	if rand() < r_cross:
		# select crossover point that is not on the end of the string
		pt = randint(1, len(p1)-2)
		# perform crossover
		c1 = np.concatenate([p1[:pt],p2[pt:]])
		c2 = np.concatenate([p2[:pt],p1[pt:]])
	return [c1, c2]

In [4]:

# mutation operator
def mutation(bitstring, r_mut):
	for i in range(len(bitstring)):
		# check for a mutation
		if rand() < r_mut:
			# flip the bit
			bitstring[i] = rand()

In [5]:
# Genetic Algorithm - Initialise Parameters
#  define the total iterations
n_iter = 100
# bits
n_bits = 4
# define the population size
n_pop = 10
# crossover rate
r_cross = 0.9
# mutation rate
r_mut = 1.0 / float(n_bits)

# Genetic Algorithm - Set up population
pop = rand(n_pop, n_bits)
best_eval = 1
best = pop[0]
scores = np.zeros(len(pop[:,1]))

ANN Functions

In [6]:
# draw number

def draw_number(image_num):
    # Open the 100 training samples in read mode
    data_file = open('mnist_train_100.csv','r')

    # Read all of the lines from the file into memory
    data_list = data_file.readlines()

    # Close the file (we are done with it)
    data_file.close()

    # Take the first line (data_list index 0, the first sample), and split it up based on the commas
    # all_values now contains a list of [label, pixel 1, pixel 2, pixel 3, ... , pixel 784]
    all_values = data_list[image_num].split(',')

    # Take the long list of pixels (but not the label), and reshape them to a 2D array of pixels
    image_array = np.asfarray(all_values[1:]).reshape((28, 28))

    # PLot this 2D array as an image, use the grey colour map and don't interpolate
    plt.imshow(image_array, cmap='Greys', interpolation='None')
    plt.show()

In [7]:
# Neural network class definition
class NeuralNetwork:
    # Init the network, this gets run whenever we make a new instance of this class
    def __init__(self, input_nodes, hidden_nodes, output_nodes, learning_rate):
        # Set the number of nodes in each input, hidden and output layer
        self.i_nodes = input_nodes
        self.h_nodes = hidden_nodes
        self.o_nodes = output_nodes

        # Weight matrices, with (input -> hidden) and who (hidden -> output)
        self.wih = numpy.random.normal(0.0, pow(self.h_nodes, -0.5), (self.h_nodes, self.i_nodes))
        self.who = numpy.random.normal(0.0, pow(self.o_nodes, -0.5), (self.o_nodes, self.h_nodes))

        # Set the learning rate
        self.lr = learning_rate

        # Set the activation function, the logistic sigmoid
        self.activation_function = lambda x: scipy.special.expit(x)

    # Train the network using back-propagation of errors
    def train(self, inputs_list, targets_list):
        # Convert inputs into 2D arrays
        inputs_array = numpy.array(inputs_list, ndmin=2).T
        targets_array = numpy.array(targets_list, ndmin=2).T

        # Calculate signals into hidden layer
        hidden_inputs = numpy.dot(self.wih, inputs_array)

        # Calculate signals emerging from hidden layer
        hidden_outputs = self.activation_function(hidden_inputs)

        # Calculate signals into final output layer
        final_inputs = numpy.dot(self.who, hidden_outputs)

        # Calculate the signals emerging from final output layer
        final_outputs = self.activation_function(final_inputs)

        # Current error is (target - actual)
        output_errors = targets_array - final_outputs

        # Hidden layer errors are the output errors, split by the weights, recombined at hidden nodes
        hidden_errors = numpy.dot(self.who.T, output_errors)

        # Update the weights for the links between the hidden and output layers
        self.who += self.lr*numpy.dot((output_errors*final_outputs*(1.0 - final_outputs)), numpy.transpose(hidden_outputs))
        # Update the weights for the links between the input and hidden layers
        self.wih += self.lr*numpy.dot((hidden_errors*hidden_outputs*(1.0 - hidden_outputs)), numpy.transpose(inputs_array))

    # Query the network
    def query(self, inputs_list):
        # Converts the inputs list into a 2D array
        inputs_array = numpy.array(inputs_list, ndmin=2).T

        # Calcualte the signals into hidden layer
        hidden_inputs = numpy.dot(self.wih, inputs_array) 

        # Calculate output from the hidden layer
        hidden_outputs = self.activation_function(hidden_inputs)

        # Calculate signals into final layer
        final_inputs = numpy.dot(self.who, hidden_outputs)

        # Calculate outputs from the final layer
        final_outputs = self.activation_function(final_inputs)
        #print(final_outputs)
        return final_outputs


In [8]:
# Load the MNIST 100 training samples CSV file into a list
training_data_file = open("mnist_train_100.csv",'r')
training_data_list = training_data_file.readlines()
training_data_file.close()

# Load the MNIST test samples CSV file into a list
test_data_file = open("mnist_test_10.csv",'r')
test_data_list = test_data_file.readlines()
test_data_file.close()

for n_round in range(n_iter):
    print("round ", n_round)
    for gene in range(n_pop):
        print("gene ", gene)

        # --- create model --------------------------------------------------
        input_nodes = 784
        hidden_nodes = round(500*pop[gene][0])
        output_nodes = 10 
        learning_rate = pop[gene][1]

        n = NeuralNetwork(input_nodes, hidden_nodes, output_nodes, learning_rate)
        
        # --- training ------------------------------------------------------
        # Train the neural network on each training sample
        for record in training_data_list:
            # Split the record by the commas
            all_values = record.split(',')
            # Scale and shift the inputs from 0..255 to 0.01..1
            inputs = (np.asfarray(all_values[1:])/255.0*0.99) + 0.01
            # Create the target output values (all 0.01, except the desired label which is 0.99)
            targets = np.zeros(output_nodes) + 0.01
            # All_values[0] is the target label for this record
            targets[int(all_values[0])] = 0.99
            # Train the network
            for i in range(round(500*pop[gene][2])):
                n.train(inputs,targets)
        pass

        # Scorecard list for how well the network performs, initially empty
        scorecard = []

        # Loop through all of the records in the test data set
        for record in test_data_list:
            # Split the record by the commas
            all_values = record.split(',')
            # The correct labelis the first value
            correct_label = int(all_values[0])
            print(correct_label, "Correct label")
            # Scale and shift the inputs
            inputs = (np.asfarray(all_values[1:])/255.0*0.99)+0.01
            # Query the network
            outputs = n.query(inputs)
            # The index of the highest value output corresponds to the label
            label = np.argmax(outputs)
            print(label,"Network label")
            # Append either a 1 or a 0 to the scorecard list
            if (label == correct_label):
                scorecard.append(1)
            else:
                scorecard.append(0)
                pass
            pass

        # Calculate the performance score, the fraction of correct answers
        scorecard_array = np.asarray(scorecard)
        # Exercise 2
        #print("Performance = ", (scorecard_array.sum()/scorecard_array.size)*100, '%')

        # scores[gene] = np.mean(gene_score/5)
        ScoreANN = (scorecard_array.sum()/scorecard_array.size)*100
        scores[gene] = ScoreANN
    
    # check for new best solution
    for p in range(n_pop):
        if scores[p] > best_eval:
            best = pop[p][:]
            best_eval = scores[p]
            print(n_round)
            print(best)
            print(best_eval)
    
    # Select Parents
    # selected = rand(n_pop, n_bits)
    # for q in range(n_pop):
    #     selected[q,:] = selection(selected[q,:], pop[q,:], scores, 3)
    selected = [selection(pop, scores) for _ in range(n_pop)]

    children = list()
    # Create next generation
    for r in range(0, n_pop, 2):
        # get selected parents in pairs
        p1, p2 = selected[r], selected[r+1]
        # crossover and mutation
        for c in crossover(p1, p2, r_cross):
            # mutation
            mutation(c, r_mut)
            # store for next generation
            children.append(c)
    # replace population
    pop = children

round  0
gene  0
7 Correct label
7 Network label
2 Correct label
6 Network label
1 Correct label
1 Network label
0 Correct label
0 Network label
4 Correct label
4 Network label
1 Correct label
1 Network label
4 Correct label
4 Network label
9 Correct label
3 Network label
5 Correct label
4 Network label
9 Correct label
7 Network label
gene  1
7 Correct label
7 Network label
2 Correct label
2 Network label
1 Correct label
1 Network label
0 Correct label
0 Network label
4 Correct label
4 Network label
1 Correct label
1 Network label
4 Correct label
4 Network label
9 Correct label
3 Network label
5 Correct label
4 Network label
9 Correct label
7 Network label
gene  2
7 Correct label
7 Network label
2 Correct label
2 Network label
1 Correct label
1 Network label
0 Correct label
0 Network label
4 Correct label
4 Network label
1 Correct label
1 Network label
4 Correct label
4 Network label
9 Correct label
3 Network label
5 Correct label
1 Network label
9 Correct label
7 Network label
gene  3

KeyboardInterrupt: 

In [9]:
print(best_eval)
print(best)

# 70.0
# [h_nodes = 250*0.65609996 lr = 0.41411952 epochs = 500*0.00408567]

70.0
[0.65609996 0.41411952 0.00408567 0.8612446 ]
